# पाठ 09 - मेटाकॉग्निशन डिझाइन पॅटर्न


## सेटअप

हे नोटबुक Microsoft Agent Framework वापरून Metacognition डिझाइन पॅटर्न दाखवते.

**पूर्वआवश्यकता:**
- Azure OpenAI तैनाती पर्यावरण चलांद्वारे कॉन्फिगर केलेली
- Azure CLI प्रमाणीकृत (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## मेटाकॉग्निशन म्हणजे काय?

मेटाकॉग्निशन म्हणजे **विचारांविषयी विचार करणे**. एआय एजंट्सच्या संदर्भात, याचा अर्थ असे एजंट तयार करणे आहे जे करू शकतात:

- **स्वत:चे चिंतन करणे** त्यांच्या स्वतःच्या आउटपुट आणि तर्क प्रक्रियेवर
- **त्रुटी ओळखणे** आणि गुप्तपणे अयशस्वी होण्याऐवजी योग्य प्रकारे पुनर्प्राप्त होणे
- **मूल्यांकन करणे** की त्यांच्या प्रतिसादांमध्ये पूर्णता आणि उपयुक्तता आहे का
- **अनुकूल करणे** जेव्हा प्रारंभिक पध्दत काम करत नाही तेव्हा त्यांच्या धोरणात बदल करणे (उदा., बॅकअप प्रणालीकडे मागे वळणे)

एक मेटाकॉग्निशन एजंट फक्त प्रश्नांची उत्तरे देत नाही — तो आपल्या स्वतःच्या कामगिरीवर लक्ष ठेवतो आणि त्वरित समायोजित करतो.


## प्राथमिक आणि बॅकअप साधने

एक सामान्य मेटाकॉग्निशन नमुना म्हणजे **बॅकअप धोरण**. एजंट प्रथम प्राथमिक साधन वापरून पाहतो; ते अयशस्वी झाल्यास (उदा., 404 त्रुटी), एजंट त्या अयशस्वीतेची ओळख करून घेतो आणि पारदर्शकपणे बॅकअप साधनाकडे स्विच करतो.

हे प्रत्यक्ष जगातील प्रणालींना प्रतिबिंबित करते जिथे प्राथमिक सेवा उपलब्ध नसू शकतात आणि एजंट्सना पर्यायी मार्ग निवडण्यापूर्वी समस्या स्वतः निदान करावी लागते.

खाली आम्ही दोन फ्लाइट शोध साधने परिभाषित करतो:
- **प्राथमिक** — Paris, Tokyo, आणि Barcelona यांना समाविष्ट करते
- **बॅकअप** — Berlin, Sydney, आणि New York City यांना समाविष्ट करते


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## त्रुटी पुनर्प्राप्तीसह स्वयं-प्रतिबिंब करणारा एजंट

खालील एजंटला असे निर्देश देण्यात आले आहेत की तो प्रथम प्राथमिक उड्डाण प्रणाली प्रयत्न करेल, अपयश ओळखेल, आणि पारदर्शकपणे बॅकअप प्रणालीकडे परत जाईल. प्रत्येक प्रतिसादानंतर तो संक्षेपात स्वतःचे मुल्यांकन करतो की त्याने वापरकर्त्याच्या प्रश्नाचे पूर्णपणे उत्तर दिले आहे का.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## स्व-मूल्यांकन नमुना

मेटाकॉग्निशनचा आणखी एक पैलू म्हणजे **स्व-मूल्यांकन**: एक वेगळा एजंट (किंवा तोच एजंट दुसऱ्या पासमध्ये) प्रतिसादाची पूर्णता, अचूकता आणि उपयुक्ततेसाठी पुनरावलोकन करतो.

खाली आपण एक `ResponseEvaluator` एजंट तयार करतो जो ट्रॅव्हल-एजंटच्या प्रतिसादांना तीन परिमाणांवर गुणांकन करतो.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## सारांश

या धड्यात आपण Microsoft Agent Framework वापरून **मेटाकॉग्निटिव्ह एजंट्स** कसे तयार करायचे हे शिकलात:

- **स्व-चिंतन**: आपल्या स्वतःच्या तर्कशक्तीवर लक्ष ठेवणारे आणि काय घडले ते पारदर्शकपणे सांगणारे एजंट.
- **फॉलबॅक्ससह त्रुटी पुनर्प्राप्ती**: एक प्राथमिक + बॅकअप टूल पॅटर्न ज्यात एजंट अपयश ओळखतो (उदा., 404 त्रुटी) आणि स्वयंचलितपणे पर्यायी स्रोत वापरून पाहतो.
- **स्व-मूल्यांकन**: एक स्वतंत्र मूल्यांकन करणारा एजंट जो प्रतिसादांना पूर्णतेसाठी, अचूकतेसाठी आणि उपयुक्ततेसाठी गुण देतो.

हे पॅटर्न एजंटांना अधिक मजबूत, पारदर्शक आणि विश्वासार्ह बनवतात — उत्पादन तैनातीसाठी अत्यंत महत्त्वाची वैशिष्ट्ये.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
अस्वीकरण:
हा दस्तऐवज AI अनुवाद सेवा Co‑op Translator (https://github.com/Azure/co-op-translator) वापरून अनुवादित केला आहे. आम्ही अचूकतेसाठी प्रयत्न करतो, परंतु कृपया लक्षात घ्या की स्वयंचलित अनुवादांमध्ये चुका किंवा अचूकतेच्या त्रुटी असू शकतात. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत म्हणून मानला जावा. महत्त्वाच्या माहितीच्या बाबतीत व्यावसायिक मानवी अनुवादाची शिफारस केली जाते. या अनुवादाच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थनिर्देशाबद्दल आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
